# Caprae Lead Prioritization & Enrichment Engine
### Interactive API & Core Pipeline Walkthrough
**Candidate:** Abhilash Maiske  
**Reference:** SaaSQuatch Leads (Path A - Quality First)

This notebook demonstrates the end-to-end Python pipeline for:
1. **Data Hygiene & RFC Validation** (Catch-all detection, domain checking)
2. **Fuzzy Deduplication & Disjoint-Set Clustering**
3. **Configurable ICP Scoring** (Seniority, Industry, Company Size)
4. **HubSpot & Salesforce Standard CRM Export**

In [1]:
import os
import sys
import pandas as pd

# Ensure backend modules can be imported
sys.path.append(os.path.abspath('./backend'))

from engine.validation import validate_lead_data
from engine.deduplication import run_deduplication
from engine.scoring import score_lead
from engine.crm_export import export_hubspot_csv, export_salesforce_csv

## 1. Load Raw Scraped Leads (Synthetic Dirty Data)

In [2]:
df_raw = pd.read_csv('data/sample_leads_raw.csv')
print(f"Loaded {len(df_raw)} raw scraped leads.")
df_raw.head(5)

## 2. Execute Data Hygiene & Validation
We detect generic role accounts (`info@`, `sales@`), malformed emails, and invalid domains, computing an objective 0-100 quality score.

In [3]:
raw_records = df_raw.to_dict(orient='records')
validated_records = []

for r in raw_records:
    val = validate_lead_data(r)
    validated_records.append({**r, **val})

df_val = pd.DataFrame(validated_records)
print("Average Data Quality Score:", df_val['data_quality_score'].mean())
df_val[['first_name', 'last_name', 'email', 'is_role_based_email', 'data_quality_score', 'quality_issues']].head(6)

## 3. Fuzzy Deduplication & Cluster Collapsing
Normalizes corporate entity legal suffixes and matches names and domains using sequence similarity.

In [4]:
deduped_records = run_deduplication(validated_records)
df_dedup = pd.DataFrame(deduped_records)

total = len(df_dedup)
dups = df_dedup['is_duplicate'].sum()
print(f"Total: {total} | Unique Leads: {total - dups} | Duplicates Collapsed: {dups}")

# View flagged duplicates and rationale
df_dedup[df_dedup['is_duplicate']][['first_name', 'last_name', 'company', 'domain', 'duplicate_reason']]

## 4. Configurable ICP Fit Scoring
Applies weighted scoring: 40% Title Seniority, 35% Industry Match, 25% Headcount Fit.

In [5]:
icp_config = {
    "weight_seniority": 40,
    "weight_industry": 35,
    "weight_size": 25,
    "target_tier1_industries": ["B2B SaaS", "Enterprise Software", "Healthcare / HealthTech", "FinTech", "Cybersecurity", "Artificial Intelligence"],
    "target_tier2_industries": ["Data & Analytics", "DevOps & Infrastructure", "Logistics & Supply Chain"],
    "min_company_size": 50,
    "max_company_size": 1000
}

scored_records = []
for lead in deduped_records:
    sc = score_lead(lead, icp_config)
    scored_records.append({**lead, **sc})

df_scored = pd.DataFrame(scored_records).sort_values(by='icp_score', ascending=False)
print(f"Tier 1 Leads Count: {sum(df_scored['icp_tier'].str.contains('Tier 1'))}")
df_scored[['first_name', 'last_name', 'title', 'company', 'icp_score', 'icp_tier', 'score_rationale']].head(8)

## 5. Export to Standard CRM Formats (HubSpot / Salesforce)

In [6]:
hubspot_csv = export_hubspot_csv(scored_records, only_unique=True)
print("HubSpot Clean CSV Preview (First 3 Lines):")
for line in hubspot_csv.strip().split('\n')[:3]:
    print(line)

# Save locally
with open('hubspot_leads_clean.csv', 'w', encoding='utf-8') as f:
    f.write(hubspot_csv)
print("\nSuccessfully exported 'hubspot_leads_clean.csv'!")